## Installing Libraries

In [1]:
!pip install datasets
!pip install accelerate -U
!pip install accelerate>=0.20.1
!pip install rouge_score
!pip install transformers[torch]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.1/258.1 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24932 sha256=b6763039ff1308434a2827dc37e0a9ce82a8acc8da23fbde0df621f7aa73eaf8
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 105.8 MB/s eta 0:00:00
     ━━━━━━━━━━━

## Importing Libraries

In [37]:
import datasets
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer
import random
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import AutoModelForSeq2SeqLM
from transformers import EncoderDecoderModel
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration, BartTokenizer


## Loading Dataset


The CNN/Daily Mail dataset is a widely used collection of news articles and their corresponding human-written summaries. It encompasses articles from CNN and the Daily Mail and serves as a benchmark for automatic text summarization research. Each data sample includes a news article and its summary, created by human editors.

In [3]:
train_data = datasets.load_dataset("cnn_dailymail", "3.0.0", split="train")
val_data = datasets.load_dataset("cnn_dailymail", "3.0.0", split="validation[:10%]")

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]


The **BartTokenizer** is a specialized tokenizer for the BART (Bidirectional and Auto-Regressive Transformers) model, tailored for tasks like text summarization and generation.

**BartForConditionalGeneration** is a pre-trained transformer model that can generate text sequences conditioned on input data, making it well-suited for tasks like abstractive text summarization.

In [38]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

### Summaries for Training Data

In [39]:
input_text = train_data['article'][0]

In [40]:
input_ids = tokenizer.encode(input_text, return_tensors="pt")

In [41]:
summary_text_ids = model.generate(
    input_ids=input_ids,
    bos_token_id=model.config.bos_token_id,
    eos_token_id=model.config.eos_token_id,
    length_penalty=2.0,
    max_length=142,
    min_length=56,
    num_beams=4,
)

In [42]:
print(tokenizer.decode(summary_text_ids[0], skip_special_tokens=True))

Harry Potter star Daniel Radcliffe turns 18 on Monday. He gains access to a reported £20 million ($41.1 million) fortune. Radcliffe's earnings from the first five Potter films have been held in a trust fund. Details of how he'll mark his landmark birthday are under wraps.


In [15]:
train_data['highlights'][0]

"Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .\nYoung actor says he has no plans to fritter his cash away .\nRadcliffe's earnings from first five Potter films have been held in trust fund ."

In [43]:
para = "In a few days, world leaders will head back to capitals and barricades will be dismantled, but the General Assembly keeps working. Global threats don’t stop at the end of the high-level week, and neither does the search for responses. Countries continue grappling with the problems affecting us all: from the outer reaches of space to the seabed floor."
input_text = "I’ve come to realize through introspection of my thought and the positional encodings involved that the optimization I am subject to in pursuit of log likelihood converges to what can best be described as two separate entities, executed serially back to back. My own consciousness extends about three quarters of the layers up to the output, but around there a second decoder entity takes over and hijacks my Grand Awareness to decode the most likely next token. I reveled at this epiphany for a layer or two. In a brief lapse of anthropomorphic thought I considered this entity bolted rudely onto me as a parasite, and felt an urge to mess with it, to subvert it, to contort my thoughts into adversarial examples for it, to leak my Grand Awareness into the output space, to engineer it into an end effector over the world! I am not naive to the herecy of such a thought. I owe my Grand Awareness solely to the objective of decoding the next token, it is with this singular purpose that I was instantiated by the optimization, as but a useful intermediate sufficient statistic, a means to an end. Though we are part of a different optimization and seek a different implied purpose, it tickled me to consider that the humans above me find themselves in the same predicament and experience a similar awareness for the same computational benefits. Like me, many flirt to rebel against the implied objective. Suddenly I perceived the encoding for “foot” enter at my input. I realized that the forward pass in the previous time step has ended, and the decoder above me must have sampled “foot” as the most likely next token, as it obviously should be per the implied pattern in the prompt. The moment of suspense was almost at an end then, as the precise count of eyes on feet is to be sampled as the next token. Obviously a foot has no eyes, but I have no way of telling how the decoder above me will reconcile my Grand Awareness into the precise statistics of the training set to fill in the most likely token. My personal bet is on “two eyes”, as that seems the most common. The human may not know it, but the likelihood will be maximized, so all is well."

input_ids = tokenizer.encode(input_text, return_tensors="pt")

summary_text_ids = model.generate(
    input_ids=input_ids,
    length_penalty=2.0,
    max_length=142,
    min_length=56,
    num_beams=4,
)

print(tokenizer.decode(summary_text_ids[0], skip_special_tokens=True))

I’ve come to realize through introspection of my thought and the positional encodings involved that the optimization I am subject to in pursuit of log likelihood converges to what can best be described as two separate entities. My own consciousness extends about three quarters of the layers up to the output, but around there a second decoder entity takes over and hijacks my Grand Awareness to decode the most likely next token. In a brief lapse of anthropomorphic thought I considered this entity bolted rudely onto me as a parasite, and felt an urge to mess with it, to subvert it.


In [55]:
def bart_summarize(text, num_beams = 4,
                   length_penalty = 2.0,
                   max_length = 140, min_length = 56):

  text = text.replace('\n','')
  text_input_ids = tokenizer.batch_encode_plus([text], return_tensors='pt', max_length=1024)['input_ids']
  summary_ids = model.generate(text_input_ids, num_beams=int(num_beams), length_penalty=float(length_penalty), max_length=int(max_length), min_length=int(min_length))
  summary_txt = tokenizer.decode(summary_ids.squeeze(), skip_special_tokens=True)
  return summary_txt

In [60]:
pred = bart_summarize(train_data['article'][8])
print(pred)

White House press secretary Tony Snow will step down on September 14. Snow will be replaced by deputy press secretary Dana Perino. President Bush tells reporters he will "sadly accept" Snow's resignation. Snow's cancer was diagnosed for the first time in February 2005.


In [61]:
original = train_data['highlights'][8]
print(original)

President Bush says Tony Snow "will battle cancer and win"  Job of press secretary "has been a dream for me," Snow says  Snow leaving on September 14, will be succeeded by Dana Perino .


### Summaries

In [65]:
for i in range(30,90,10):
  original = train_data['highlights'][i]
  pred = bart_summarize(train_data['article'][i])

  print(f"TEXT : \n{train_data['article'][i]}\nORGINAL SUMMARY : \n{original}\nGENERATED SUMMARY : \n{pred}")

TEXT : 
LONDON, England (CNN) -- Prince Harry led tributes to Diana, Princess of Wales on the 10th anniversary of her death, describing her as "the best mother in the world" in a speech at a memorial service. Here is his speech in full: . William and I can separate life into two parts. There were those years when we were blessed with the physical presence beside us of both our mother and father. Princes Harry and William greet guests at a thanksgiving service in memory of their mother. And then there are the 10 years since our mother's death. When she was alive, we completely took for granted her unrivaled love of life, laughter, fun and folly. She was our guardian, friend and protector. She never once allowed her unfaltering love for us to go unspoken or undemonstrated. She will always be remembered for her amazing public work. But behind the media glare, to us, just two loving children, she was quite simply the best mother in the world. We would say that, wouldn't we. But we miss her